In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
from pathlib import Path

ROOT = Path("..")      # the data files sit one folder above notebooks/

CAP = {"S": 1, "M": 3, "L": 5}                                    # ambulances a site of each size can hold
FIX = {"EMS": {"S": 0.0, "M": 1.0, "L": 2.0},                     # fixed cost in $m, existing EMS stations
       "OTHER": {"S": 1.5, "M": 2.5, "L": 3.5}}                   # firehouses and hospitals (conversion)
C_AMB = 0.590          # $m per ambulance
S_HOURS = 1.5          # hours an ambulance is busy per call
RHO = 0.60             # share of a period an ambulance can be busy
H = 6                  # hours in a period
BUDGET = 30.0          # $m

# 04 Optimization model

Chooses which sites to open, their size, and how many ambulances each gets, to cover as many crashes per day as possible within the budget (the MILP in the report). Solved with Gurobi.

- Reads: `a_ijt.npy`, `sites.csv`, `demand_d_it.csv`
- Needs: a Gurobi licence (the restricted pip version is too small for this model)

In [ ]:
a = np.load(ROOT / "a_ijt.npy")                  # sites x zones x periods, True if the site reaches the zone in time
sites = pd.read_csv(ROOT / "sites.csv")
d = pd.read_csv(ROOT / "demand_d_it.csv", index_col=0)
d.columns = [int(c) for c in d.columns]
dv = d.values                                    # zones x 4, average crashes per day

J, I, T = range(a.shape[0]), range(a.shape[1]), range(4)
F = {(j, k): FIX["EMS" if sites.loc[j, "site_type"] == "EMS" else "OTHER"][k] for j in J for k in CAP}
print("sites:", len(J), "| zones:", len(I), "| total daily demand:", round(dv.sum(), 2))

# only used for the maps
zones = gpd.read_file(ROOT / "zones.gpkg").set_index("zone").loc[d.index]
sp = gpd.GeoSeries(gpd.points_from_xy(sites["longitude"], sites["latitude"]), crs=4326).to_crs(32618)

In [ ]:
def build(budget=BUDGET):
    # f exists only where the site reaches the zone and the zone has demand
    pairs = [(i, j, t) for j in J for i in I for t in T if a[j, i, t] and dv[i, t] > 0]
    by_it, by_jt = defaultdict(list), defaultdict(list)
    for i, j, t in pairs:
        by_it[(i, t)].append(j)
        by_jt[(j, t)].append(i)

    m = gp.Model("ambulances")
    m.setParam("OutputFlag", 0)
    q = m.addVars(J, list(CAP), vtype=GRB.BINARY, name="q")        # site j opened at size k
    n = m.addVars(J, vtype=GRB.INTEGER, lb=0, name="n")            # ambulances at site j
    f = m.addVars(pairs, lb=0.0, name="f")                         # crashes per day served from j in zone i, period t

    cost = gp.quicksum(F[j, k] * q[j, k] for j in J for k in CAP) + C_AMB * n.sum()
    m.setObjective(f.sum(), GRB.MAXIMIZE)

    m.addConstrs((q.sum(j, "*") <= 1 for j in J))                                          # one size per site
    m.addConstrs((n[j] >= q.sum(j, "*") for j in J))                                       # an open site has an ambulance
    m.addConstrs((n[j] <= gp.quicksum(CAP[k] * q[j, k] for k in CAP) for j in J))          # capacity
    for (i, t), js in by_it.items():                                                       # can't serve more than the demand
        m.addConstr(gp.quicksum(f[i, j, t] for j in js) <= dv[i, t])
    for (j, t), is_ in by_jt.items():                                                      # ambulance time
        m.addConstr(S_HOURS * gp.quicksum(f[i, j, t] for i in is_) <= RHO * H * n[j])

    # Redundant for correctness, but it makes the LP bound much tighter and the solves much faster:
    # nothing can be served from a site that isn't open.
    opened = {j: q.sum(j, "*") for j in J}
    for (i, j, t) in pairs:
        m.addConstr(f[i, j, t] <= dv[i, t] * opened[j])

    bud = m.addConstr(cost <= budget)
    return dict(m=m, q=q, n=n, f=f, bud=bud, total=dv.sum(), npairs=len(pairs))

M = build()
print("f variables:", M["npairs"])

The extra constraint f ≤ d·(site is open) does not change the optimum. Without it the solver took minutes, or timed out, at budgets where the budget actually binds. With it the same solves take a second or so.

In [ ]:
def best_coverage(M, budget, time_limit=60, gap=0.001):
    """Re-solve the same model with a different budget. Returns (coverage share, proven optimal)."""
    M["bud"].RHS = budget
    M["m"].setParam("TimeLimit", time_limit)
    M["m"].setParam("MIPGap", gap)
    M["m"].optimize()
    cov = M["m"].ObjVal / M["total"] if M["m"].SolCount else 0.0
    return cov, M["m"].Status == GRB.OPTIMAL


def show(M, budget):
    """Table of the plan from the last solve."""
    rows = []
    for j in J:
        for k in CAP:
            if M["q"][j, k].X > 0.5:
                amb = int(round(M["n"][j].X))
                typ = sites.loc[j, "site_type"]
                if typ == "EMS":
                    change = "existing EMS station" + ("" if k == "S" else f", expanded to {k}")
                else:
                    change = "converted " + typ.lower()
                rows.append((j, typ, sites.loc[j, "facname"], k, amb, change, F[j, k], C_AMB * amb))
    res = pd.DataFrame(rows, columns=["site", "type", "name", "size", "ambulances", "what changes",
                                      "station_$m", "vehicles_$m"])
    res["total_$m"] = res["station_$m"] + res["vehicles_$m"]
    cov = M["m"].ObjVal / M["total"]
    print(f"budget ${budget:g}m | coverage {cov:.4f} | spent ${res['total_$m'].sum():.2f}m")
    print(f"sites used: {len(res)} (EMS stations {int((res['type'] == 'EMS').sum())} of 14, "
          f"firehouses {int((res['type'] == 'FIREHOUSE').sum())}, hospitals {int((res['type'] == 'HOSPITAL').sum())}) "
          f"| ambulances: {int(res['ambulances'].sum())}")
    return res


def plot_plan(res, title):
    fig, ax = plt.subplots(figsize=(5, 8))
    zones.plot(ax=ax, column=pd.Series(dv.sum(axis=1), index=zones.index), cmap="Reds", alpha=0.5)
    ax.scatter(sp[res["site"]].x, sp[res["site"]].y, s=40 + 60 * res["ambulances"].values,
               color="#1f4e79", edgecolor="white", zorder=3)
    ax.set_title(title + " (dot size = ambulances)"); ax.set_axis_off()
    plt.show()

In [ ]:
cov, ok = best_coverage(M, BUDGET)
print("coverage:", round(cov, 4), "| proven optimal:", ok)
plan = show(M, BUDGET)
plot_plan(plan, "Plan at $30m")
plan

At the full $30m we can cover 99.94% of crash demand, which is essentially the 99.9% ceiling from notebook 03. This plan should not be read as the plan to build: the objective only counts coverage, not cost, so at $30m the solver can stop anywhere that gets the maximum and the particular sites chosen are arbitrary. The budget curve and the smallest-budget plan below are more useful.

In [ ]:
rows = []
for B in [1, 2, 3, 4, 6, 8, 10, 15, 20, 30]:
    cov, ok = best_coverage(M, B)
    rows.append((B, round(cov, 4), ok))
curve = pd.DataFrame(rows, columns=["budget_$m", "coverage_share", "proven_optimal"])

plt.figure(figsize=(6, 3.5))
plt.plot(curve["budget_$m"], curve["coverage_share"], marker="o")
plt.xlabel("budget ($m)"); plt.ylabel("share of crash demand covered")
plt.title("Coverage against budget"); plt.grid(alpha=0.3)
plt.show()
curve

- Coverage climbs fast at first: 24% at $1m, 60% at $2m, 92% at $6m.
- By $10m we are at 99.3%, and after about $15m extra money buys almost nothing (99.89% at $15m, 99.94% at $20m and $30m).
- The report's $30m budget is therefore far more than is needed for this demand.

In [ ]:
def min_budget(M, target, hi=BUDGET, tol=0.05):
    """Smallest budget (within tol, in $m) whose best plan reaches the target coverage."""
    lo = 0.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        cov, _ = best_coverage(M, mid)
        if cov >= target:
            hi = mid
        else:
            lo = mid
    return hi

TARGET = 0.99
B99 = min_budget(M, TARGET)
cov, ok = best_coverage(M, B99)
print(f"smallest budget for {TARGET:.0%} coverage: about ${B99:.2f}m (coverage {cov:.4f}, proven optimal: {ok})")
plan99 = show(M, B99)
plot_plan(plan99, f"Plan at ${B99:.1f}m")
plan99

This is the leaner plan: the cheapest one the model finds that still covers 99% of crashes. It is a better starting point for a recommendation than the $30m plan, since every site in it is there for a reason.